<a href="https://colab.research.google.com/github/hasibXtreme/Triage_engine/blob/main/Medical_triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
class mixednaivebayes:
  def __init__(self,laplace_alpha=0.01):
    self.alpha = laplace_alpha


  def fit(self,X_cont,X_cat,y):
    self.classes = np.unique(y)
    n_classes = len(self.classes)
    n_samples = len(y)

    self.priors = np.array([np.sum(y==c)/float(n_samples) for c in self.classes])

    self.means = np.zeros((n_classes , X_cont.shape[1]))
    self.variences = np.zeros((n_classes,X_cont.shape[1]))
    for idx,c in enumerate(self.classes):
      x_c = X_cont[y==c]
      self.means[idx,:] =  x_c.mean(axis=0)
      self.variences[idx,:] =  x_c.var(axis=0) + 1e-9

    self.cat_prob  = []
    for feat_idx in range(X_cat.shape[1]):
      feat_col = X_cat[:,feat_idx]
      categories = np.unique(feat_col)
      prob_dict = {}

      for idx,c in enumerate(self.classes):
        feat_c = feat_col[y==c]
        num_cat = len(categories)
        total_count_c = len(feat_c)
        prob_dict[c]={}

        for cat in categories:
          count = np.sum(feat_c==cat)
          prob_dict[c][cat] = (count+self.alpha)/total_count_c+(self.alpha*num_cat)

      self.cat_prob.append(prob_dict)

  def predict(self,X_cont,X_cat):
    predictions = []

    for i in range(len(X_cont)):
      x_cont_i=X_cont[i]
      x_cat_i=X_cat[i]


      log_posteriors=[]

      for idx,c in enumerate(self.classes):
        log_score = np.log(self.priors[idx])

        mean = self.means[idx]
        varience= self.variences[idx]
        log_pdf = -0.5* np.sum(np.log(2*np.pi*varience))-0*5*np.sum(((x_cont_i-mean)**2)/varience)
        log_score+=log_pdf


        for feat_idx,val in enumerate(x_cat_i):
          prob = self.cat_prob[feat_idx][c].get(val,1e-9)
          log_score+=np.log(prob)

        log_posteriors.append(log_score)

      predictions.append(self.classes[np.argmax(log_posteriors)])

    return np.array(predictions)

In [ ]:
df = pd.read_csv('heart.csv')
continuous_features = ['trtbps', 'chol', 'thalachh', 'oldpeak', 'age']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exng', 'slp', 'caa', 'thall']

X_cont = df[continuous_features].values
X_cat = df[categorical_features].values
y = df['output'].values

x_cont_train,x_cont_test,x_cat_train,x_cat_test,y_train,y_test = train_test_split(X_cont,X_cat,y,test_size=0.2,random_state=42,stratify=y)

model = mixednaivebayes(laplace_alpha=0.01)
model.fit(x_cont_train,x_cat_train,y_train)

y_pred = model.predict(x_cont_test,x_cat_test)

print("======classification report=========")

print(classification_report(y_test,y_pred,target_names=['low risk (0)', 'high risk (1)']))

======classification report=========
               precision    recall  f1-score   support

 low risk (0)       0.94      0.57      0.71        28
high risk (1)       0.73      0.97      0.83        33

     accuracy                           0.79        61
    macro avg       0.83      0.77      0.77        61
 weighted avg       0.83      0.79      0.78        61



In [ ]:
df = pd.read_csv('heart.csv')
y = df['output']
x = df.drop(columns=['output'])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.tree import DecisionTreeClassifier,export_text
dt_model = DecisionTreeClassifier(max_depth=4,random_state=42)
dt_model.fit(x_train,y_train)

dt_preds = dt_model.predict(x_test)
print(export_text(dt_model,feature_names=list(x.columns)))
print(classification_report(y_test,dt_preds,target_names=['low risk(0)','high risk(1)']))

|--- cp <= 0.50
|   |--- oldpeak <= 0.70
|   |   |--- caa <= 0.50
|   |   |   |--- age <= 41.50
|   |   |   |   |--- class: 0
|   |   |   |--- age >  41.50
|   |   |   |   |--- class: 1
|   |   |--- caa >  0.50
|   |   |   |--- sex <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- sex >  0.50
|   |   |   |   |--- class: 0
|   |--- oldpeak >  0.70
|   |   |--- thall <= 2.50
|   |   |   |--- slp <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- slp >  1.50
|   |   |   |   |--- class: 1
|   |   |--- thall >  2.50
|   |   |   |--- class: 0
|--- cp >  0.50
|   |--- thall <= 2.50
|   |   |--- thall <= 1.50
|   |   |   |--- age <= 54.50
|   |   |   |   |--- class: 1
|   |   |   |--- age >  54.50
|   |   |   |   |--- class: 0
|   |   |--- thall >  1.50
|   |   |   |--- oldpeak <= 3.55
|   |   |   |   |--- class: 1
|   |   |   |--- oldpeak >  3.55
|   |   |   |   |--- class: 0
|   |--- thall >  2.50
|   |   |--- thalachh <= 142.50
|   |   |   |--- trtbps <= 135.00
|   |   |   |   |---

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)
rf_model.fit(x_train,y_train)
rf_preds =  rf_model.predict(x_test)

print(classification_report(y_test,rf_preds,target_names=['low risk(0)','high risk(1)']))


              precision    recall  f1-score   support

 low risk(0)       0.90      0.64      0.75        28
high risk(1)       0.76      0.94      0.84        33

    accuracy                           0.80        61
   macro avg       0.83      0.79      0.79        61
weighted avg       0.82      0.80      0.80        61



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

xgb_model = GradientBoostingClassifier()
xgb_model.fit(x_train,y_train)

xgb_preds = xgb_model.predict(x_test)

print(classification_report(y_test,xgb_preds,target_names=['low risk(0)','high risk(1)']))

              precision    recall  f1-score   support

 low risk(0)       0.87      0.71      0.78        28
high risk(1)       0.79      0.91      0.85        33

    accuracy                           0.82        61
   macro avg       0.83      0.81      0.81        61
weighted avg       0.83      0.82      0.82        61



In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# 1. MANDATORY: Standardize Features (Mean = 0, Variance = 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

# 2. Initialize & Train (RBF Kernel with Probability Estimation)
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 3. Inference
svm_preds = svm_model.predict(X_test_scaled)
print(classification_report(y_test,svm_preds,target_names=['low risk(0)','high risk(1)']))

              precision    recall  f1-score   support

 low risk(0)       0.90      0.68      0.78        28
high risk(1)       0.78      0.94      0.85        33

    accuracy                           0.82        61
   macro avg       0.84      0.81      0.81        61
weighted avg       0.83      0.82      0.82        61

